# Capítulo 1 · Laboratório 1: o seu primeiro notebook
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026** (o mesmo corte do livro). Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap01.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Se você mudar a data `FIM`, apague o arquivo `dados_cap01.pkl` para o notebook baixar os dados novos; aí os números mudam: é esse o objetivo.

Quatro partes:
1. o primeiro gráfico (Ibovespa e PETR4);
2. aleatório ou real? (a moeda de Malkiel e o teste de Lo);
3. o primeiro scanner (máxima de 52 semanas, sem abrir gráfico);
4. petróleo negativo (WTI, abril de 2020).

In [ ]:
!pip install -q yfinance

In [ ]:
import logging, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

logging.getLogger('yfinance').setLevel(logging.CRITICAL)   # esconde avisos de código sem dados
FIM = '2026-10-01'      # o yfinance não inclui o dia final: dados até 30/09/2026
pc = lambda x, k=1: f'{x:.{k}%}'.replace('.', ',')
nr = lambda x, k=2: f'{x:,.{k}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

def baixa(ticker, inicio):
    d = yf.download(ticker, start=inicio, end=FIM, auto_adjust=False, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=['Close'])


def confere(baixados):
    '''O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local.'''
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f'Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou'
                               ' limitando pedidos. Espere alguns minutos e rode a célula de novo.')


CACHE = 'dados_cap01.pkl'   # cópia local: a primeira execução baixa do Yahoo Finance e grava aqui; as seguintes leem daqui
D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for t, ini in [('^BVSP', '2006-01-01'), ('PETR4.SA', '2024-01-01'), ('CL=F', '2020-02-01')]:
    if t not in D:
        D[t] = baixa(t, ini)
confere({t: D[t] for t in ['^BVSP', 'PETR4.SA', 'CL=F']})
pd.to_pickle(D, CACHE)
print({k: len(v) for k, v in D.items() if isinstance(v, pd.DataFrame)})

## Parte 1. O primeiro gráfico
Candle branco = alta (fechou acima da abertura); preto = baixa.

In [ ]:
def candles(ax, d, largura=0.6):
    for i, (o, h, l, c) in enumerate(d[['Open', 'High', 'Low', 'Close']].to_numpy()):
        ax.plot([i, i], [l, h], color='black', lw=0.6)
        ax.add_patch(plt.Rectangle((i - largura / 2, min(o, c)), largura, max(abs(c - o), 1e-9),
                                   facecolor='white' if c >= o else 'black', edgecolor='black', lw=0.6))
    passo = max(len(d) // 6, 1)
    ax.set_xticks(range(0, len(d), passo))
    ax.set_xticklabels([x.strftime('%d/%m/%y') for x in d.index[::passo]])

fig, axs = plt.subplots(2, 1, figsize=(10, 7))
for ax, t in zip(axs, ['^BVSP', 'PETR4.SA']):
    candles(ax, D[t].iloc[-90:])
    ax.set_title(t)
plt.tight_layout(); plt.show()
print('Ibovespa, último fechamento:', nr(D['^BVSP']['Close'].iloc[-1], 0), 'pontos')

## Parte 2. Aleatório ou real?
**A moeda de Malkiel (1973):** a ação começa em US$ 50 e, a cada dia, sobe ou cai meio ponto no cara ou coroa.

**O teste de Hasanhodzic, Lo e Viola (2010):** pegamos os retornos diários verdadeiros de PETR4 e sorteamos outra ordem. O retorno total e a volatilidade ficam iguais; só a ordem muda. Troque a `semente` para gerar novos desafios.

In [ ]:
semente = 1973                           # 1973: ano do livro de Malkiel. Troque para ter outro desafio
rng = np.random.default_rng(semente)
moeda = 50 + np.cumsum(rng.choice([0.5, -0.5], size=250))
real = D['PETR4.SA']['Close'].loc['2025-10-01':].copy()
ret = np.log(real).diff().dropna().to_numpy()
embaralhado = real.iloc[0] * np.exp(np.concatenate([[0], np.cumsum(rng.permutation(ret))]))
ORDEM = rng.permutation(['real', 'embaralhado'])
SERIES = {'real': real.to_numpy(), 'embaralhado': embaralhado}

fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
axs[0].plot(moeda, color='black'); axs[0].set_title('A ação de Malkiel (cara ou coroa)')
for ax, rot, chave in zip(axs[1:], 'AB', ORDEM):
    ax.plot(SERIES[chave], color='black'); ax.set_title(f'Gráfico {rot}')
plt.tight_layout(); plt.show()
print(f'Moeda: de US$ 50,00 a US$ {nr(moeda[-1])}; máxima {nr(moeda.max())}, mínima {nr(moeda.min())}')

Pense antes de rodar a próxima célula: qual dos dois, A ou B, é PETR4 de verdade?

In [ ]:
print(f'Resposta: A = {ORDEM[0]}, B = {ORDEM[1]}')
r_real, r_emb = np.diff(np.log(SERIES['real'])), np.diff(np.log(SERIES['embaralhado']))
print(f'Retorno total: {pc(np.exp(r_real.sum()) - 1)} x {pc(np.exp(r_emb.sum()) - 1)}')
print(f'Volatilidade diária: {pc(r_real.std(), 2)} x {pc(r_emb.std(), 2)}')
ac = lambda x: pd.Series(np.abs(x)).autocorr(1)
print(f'Agrupamento de volatilidade (autocorrelação do |retorno|): real {nr(ac(r_real))} x embaralhado {nr(ac(r_emb))}')

## Parte 3. O primeiro scanner
Varre a lista e mostra as ações a menos de 3% da máxima de 52 semanas. Nenhum gráfico foi aberto. Troque a lista à vontade; código que não existir é ignorado.

In [ ]:
LISTA_B3 = ['PETR4', 'PETR3', 'VALE3', 'ITUB4', 'BBDC4', 'BBAS3', 'B3SA3', 'ABEV3', 'WEGE3', 'SUZB3',
            'GGBR4', 'PRIO3', 'RADL3', 'EQTL3', 'VIVT3', 'LREN3', 'EMBJ3', 'CSAN3', 'RAIL3', 'CMIG4',
            'ITSA4', 'BPAC11', 'SBSP3', 'RENT3', 'MBRF3', 'UGPA3', 'CPLE3', 'ENEV3', 'TIMS3', 'KLBN11',
            'HAPV3', 'RDOR3', 'TOTS3', 'CSNA3', 'USIM5', 'GOAU4', 'BRAP4', 'CMIN3', 'BBSE3', 'CXSE3',
            'SANB11', 'BEEF3', 'AZZA3', 'MGLU3', 'ASAI3', 'VBBR3', 'RECV3', 'BRAV3', 'EGIE3', 'TAEE11',
            'ALUP11', 'CPFE3', 'FLRY3', 'HYPE3', 'MULT3', 'CYRE3', 'MRVE3', 'DIRR3', 'YDUQ3', 'COGN3',
            'SMTO3', 'SLCE3', 'POMO4', 'IRBR3', 'PSSA3']
scan = D.get('scanner', {})
for t in LISTA_B3:
    if t not in scan:
        try:
            d = baixa(t + '.SA', '2025-06-01')
            if len(d) > 200:
                scan[t] = d
        except Exception:
            pass
D['scanner'] = scan
pd.to_pickle(D, CACHE)                   # guarda o scanner junto com os outros dados
linhas = []
for t in LISTA_B3:
    if t not in scan:
        continue
    d = scan[t].loc[:FIM]
    ult = d.index[-1]
    if ult < pd.Timestamp(FIM) - pd.Timedelta(days=8):
        continue
    jan = d.loc[ult - pd.Timedelta(days=365):]
    linhas.append({'ação': t, 'fechamento': d['Close'].iloc[-1], 'máxima 52 sem.': jan['High'].max(),
                   'distância': d['Close'].iloc[-1] / jan['High'].max() - 1})
SCAN = pd.DataFrame(linhas).sort_values('distância', ascending=False).reset_index(drop=True)
print(f'{len(SCAN)} ações varridas em {FIM}; perto da máxima (até 3%):')
SCAN[SCAN['distância'] >= -0.03]

## Parte 4. Petróleo negativo
Em 20/04/2020, o contrato de maio do WTI fechou a US$ -37,63. A série contínua `CL=F` troca de contrato no vencimento: no dia seguinte já mostra o contrato de junho (veja o Anexo II).

In [ ]:
w = D['CL=F'].loc['2020-03-02':'2020-05-29']
fig, ax = plt.subplots(figsize=(10, 4)); candles(ax, w); ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('US$ por barril'); plt.tight_layout(); plt.show()
dia = w['Close'].idxmin()
print(f"Menor fechamento: US$ {nr(w['Close'].min())} em {dia:%d/%m/%Y}; dia seguinte: US$ {nr(w['Close'].shift(-1).loc[dia])}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a Parte 2. Mostre como `rng.choice([0.5, -0.5], size=250)` lança a moeda de Malkiel e por que `rng.permutation(ret)` mantém o retorno total e a volatilidade de PETR4 e só muda a ordem." Confira com a segunda célula da Parte 2, que imprime o retorno total e a volatilidade diária das duas séries: os números têm de ser iguais. Com a `semente` 1973, a primeira célula da Parte 2 imprime a moeda indo de US$ 50,00 a US$ 51,00, com máxima de 60,00 e mínima de 50,00, como no livro.
2. **Adaptar:** "Acrescente [ATIVO] à lista `LISTA_B3` da Parte 3, com o código da B3 sem o .SA. Não mude a janela de 52 semanas nem o limite de 3%." Antes de olhar o ativo novo, confira se, com `FIM` em 2026-10-01, a Parte 3 continua listando ASAI3, ENEV3, FLRY3 e BPAC11, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Troque a `semente` da Parte 2 e mostre os gráficos A e B para um amigo. Quantos ele acerta em 10 rodadas?
2. No scanner, troque 3% por 1% e a máxima de 52 semanas pela de 20 dias.
3. Mude `FIM` para hoje e compare a lista com a do livro.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.